# Student evaluation: DeepSeek V4 Flash 0731 / Baidu FP8

This notebook evaluates GPQA student performance through OpenRouter. Requests are pinned to the exact `baidu/fp8` endpoint with provider fallbacks disabled. Every run is saved in a new folder under `outputs/`.

In [ ]:
from pathlib import Path
import json
import sys

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'content conditions').is_dir() and (candidate / 'student_eval').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the project root')

REPO_ROOT = find_repo_root()
EXPERIMENT_DIR = REPO_ROOT / 'experiments' / 'api' / 'deepseek-v4-flash-0731_baidu-fp8'
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from student_eval import CONDITION_FILES, ExperimentConfig, postprocess_run, preview_task, run_experiment
print('Project root:', REPO_ROOT)

## Experiment variables

`NUM_ROWS` is the number of aligned questions **per condition**. Use `None` for all 448 rows. Because source CSV row orders differ, questions are sorted and aligned by `id` before slicing.

In [ ]:
MODEL = 'deepseek/deepseek-v4-flash-0731'
PROVIDER = 'baidu/fp8'
TARGET_CONDITIONS = [0]        # Any subset of 0..6
NUM_ROWS = 1                   # Rows per condition; None means all rows
START_ROW = 0
CONCURRENCY = 50
TEMPERATURE = 0.0
MAX_TOKENS = 4096             # Includes the model's internal reasoning budget
REASONING_ENABLED = False     # Avoid Baidu's empty-final reasoning sink
REASONING_EFFORT = 'low'
FINAL_ANSWER_RETRIES = 2       # Retry empty/truncated finals with a larger budget
MAX_RECOVERY_TOKENS = 8192

CONFIG = ExperimentConfig(
    model=MODEL,
    provider=PROVIDER,
    condition_ids=tuple(TARGET_CONDITIONS),
    num_rows=NUM_ROWS,
    start_row=START_ROW,
    concurrency=CONCURRENCY,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    reasoning_enabled=REASONING_ENABLED,
    reasoning_effort=REASONING_EFFORT,
    final_answer_retries=FINAL_ANSWER_RETRIES,
    max_recovery_tokens=MAX_RECOVERY_TOKENS,
)

for condition_id in TARGET_CONDITIONS:
    print(f'{condition_id}: {CONDITION_FILES[condition_id]}')

## Preview one request

This does not call the API or spend credits. The reference answer is displayed separately for validation and is never inserted into the model prompt.

In [ ]:
preview = preview_task(REPO_ROOT, CONFIG)
print('Request key:', preview['request_key'])
print('Condition file:', preview['condition_file'])
print('Reference answer (not sent):', preview['reference_answer'])
print('\n--- Prompt sent to the model ---\n')
print(preview['prompt'])

## Run and save

The API key is read from `OPENROUTER_API_KEY` in the project `.env`. It is not printed or written to the run folder.

In [ ]:
run_dir, results, summary = run_experiment(REPO_ROOT, EXPERIMENT_DIR, CONFIG)
print('Saved run:', run_dir)
print(json.dumps(summary, indent=2))

## Result preview

In [ ]:
for result in results[:5]:
    print(json.dumps({
        'request_key': result['request_key'],
        'prediction': result.get('prediction'),
        'reference_answer': result['answer_key'],
        'is_correct': result.get('is_correct'),
        'reason': result.get('reason'),
        'actual_provider': result.get('actual_provider'),
        'usage': result.get('usage'),
        'latency_seconds': result.get('latency_seconds'),
        'error': result.get('error'),
    }, ensure_ascii=False, indent=2))

## Postprocess saved failures without API calls

This conservatively repairs only rows whose saved raw response contains an explicit JSON-style choice. Original files are never overwritten.

In [ ]:
post_results, post_summary, post_report, repaired_summary = postprocess_run(run_dir)
print('Postprocessed results:', post_results)
print(json.dumps(repaired_summary, indent=2))